In [1]:
import sys

import polars as pl
import torch

from modeling_module.data_loader.MultiPartDataModule import MultiPartDataModule
from modeling_module.training.config import TrainingConfig
from modeling_module.training.model_trainers.total_train import run_total_train_monthly
from modeling_module.utils.checkpoint import save_model_dict, load_model_dict

'''
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
https://developer.nvidia.com/cuda-12-8-0-download-archive
'''

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/data_research/raw_data/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/research/raw_data/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

In [2]:
target_dyn_demand_monthly = pl.read_parquet(DIR + 'target_dyn_demand_monthly.parquet')

In [3]:
from modeling_module.models.PatchTST.supervised.backbone import SupervisedBackbone
from modeling_module.models.PatchTST.common.configs import PatchTSTConfigMonthly
cfg = PatchTSTConfigMonthly(
    c_in=1,
    target_dim=1,
    lookback=24,
    horizon=6,
    patch_len=4,
    stride=2,
)

model = SupervisedBackbone(cfg)
x = torch.randn(8, cfg.c_in, cfg.lookback)
out = model(x)

print(out.shape)  # [8, L_tok, d_model]

torch.Size([8, 11, 128])


In [4]:
plan_yyyymm = 201801
lookback = 48
horizon = 12

data_module = MultiPartDataModule(
    target_dyn_demand_monthly,
    lookback = lookback,
    horizon = horizon,
    batch_size = 64,
    val_ratio = 0.2,
    is_running = False
)
train_loader = data_module.get_train_loader()
val_loader = data_module.get_val_loader()

In [5]:
model_dict = run_total_train_monthly(train_loader, val_loader, lookback = lookback, horizon = horizon)

PatchMixer Quantile


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/amp/grad_scaler.py:136: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


Epoch 1/1 | LR 0.000976 | Train 27.120451 | Val 26.563509
Titan Patch
Epoch 1/1 | LR 0.000976 | Train 158.129630 | Val 155.421214
PatchTST Base
Epoch 1/1 | LR 0.000976 | Train 39.346629 | Val 39.990859
PatchTST Quantile
Epoch 1/1 | LR 0.000976 | Train 38.755802 | Val 39.612569


In [ ]:
from modeling_module.utils.exogenous_utils import calendar_cb
from modeling_module.models.PatchTST.common.configs import PatchTSTConfigMonthly
from modeling_module.models.Titan.common.configs import TitanConfigMonthly, TitanConfigPatchMonthly
from modeling_module.models.PatchMixer.common.configs import PatchMixerConfigMonthly

save_dir = DIR + 'fit'
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

pm_base_config = PatchMixerConfigMonthly(
        device = device,
        loss_mode = 'point',
        point_loss = 'mae'
    )

pm_quantile_config = PatchMixerConfigMonthly(
    device = device,
    loss_mode = 'quantile',
    quantiles = (0.1, 0.5, 0.9)
)

ti_config = TitanConfigMonthly(
        device = device,
        loss_mode = 'point',
        point_loss = 'mae'
    )

ti_patch_config = TitanConfigPatchMonthly(
    device = device,
    loss_mode = 'point',
    point_loss = 'mae'
)

pt_config = PatchTSTConfigMonthly(
        device = device,
        loss_mode = 'auto',
        quantiles = (0.1, 0.5, 0.9)
    )

cfg_map = {
    "PatchMixer Base": pm_base_config,
    "PatchMixer Quantile": pm_quantile_config,
    "Titan Base": ti_config,
    "Titan LMM": ti_config,
    "Titan Seq2Seq": ti_config,
    "Titan Patch": ti_patch_config,
    "PatchTST Base": pt_config,
    "PatchTST Quantile": pt_config
}

builder_key_by_name = {
  "PatchMixer Base": "patchmixer_base",
  "PatchMixer Quantile": "patchmixer_quantile",
  "Titan Base": "titan_base",
  "Titan LMM": "titan_lmm",
  "Titan Seq2Seq": "titan_seq2seq",
  "Titan Patch": "titan_patch",
  "PatchTST Base": "patchtst_base",
  "PatchTST Quantile": "patchtst_quantile",
}
save_index = save_model_dict(model_dict, save_dir, cfg_by_name = cfg_map, builder_key_by_name=builder_key_by_name)

# Load
from modeling_module.models.model_builder import (
    build_patch_mixer_base, build_patch_mixer_quantile,
    build_titan_base, build_titan_lmm, build_titan_seq2seq,
    build_patchTST_base, build_patchTST_quantile, build_titan_patch,
)

builders = {
    # "patchmixer_base": lambda cfg: build_patch_mixer_base(cfg or PatchMixerConfigMonthly()),
    "patchmixer_quantile": lambda cfg: build_patch_mixer_quantile(cfg or PatchMixerConfigMonthly()),
    # "titan_base": lambda cfg: build_titan_base(cfg or TitanConfigMonthly()),
    # "titan_lmm": lambda cfg: build_titan_lmm(cfg or TitanConfigMonthly()),
    # "titan_seq2seq": lambda cfg: build_titan_seq2seq(cfg or TitanConfigMonthly()),
    'titan_patch': lambda cfg: build_titan_patch(cfg or TitanConfigPatchMonthly()),
    "patchtst_base": lambda cfg: build_patchTST_base(cfg or PatchTSTConfigMonthly()),
    "patchtst_quantile": lambda cfg: build_patchTST_quantile(cfg or PatchTSTConfigMonthly()),
}
loaded = load_model_dict(save_dir, builders, device = device)



In [ ]:
%load_ext autoreload
%autoreload 2

import importlib
import modeling_module.utils.plot_utils as plot_utils
importlib.reload(plot_utils)

plot_utils.plot_120_months_many(
    loaded, val_loader, device = device, use_truth = True,
    max_plots = 100, show = True, future_exo_cb=calendar_cb
)